# Multilingual Health Q&A - Starter Notebook

**Challenge:** Multilingual Health Question Answering in Low-Resource African Languages

This notebook provides two baselines for the challenge:

| Baseline | Approach | Speed | Quality |
|---|---|---|---|
| **Baseline 1** | TF-IDF retrieval | Very fast | Low |
| **Baseline 2** | Multilingual LLM (mT5 / NLLB) | Moderate | Higher |

**Task:** Given a health question in one of five African languages (Amharic, Luganda, Akan, Swahili, or English), generate a fluent and accurate answer in the **same language**.

**Evaluation metrics:**
- `TargetRLF1` - ROUGE-L F1 score
- `TargetR1F1` - ROUGE-1 F1 score
- `TargetLLM`  - LLM-as-a-Judge score

> **Note:** All three submission columns should contain the same generated answer for each row. The platform computes all three metrics from that single answer.

## 1 - Install and Import Packages

In [ ]:
# Install required packages
!pip install -q scikit-learn pandas numpy rouge-score
!pip install -q transformers sentencepiece accelerate torch datasets peft

print('OK Packages installed')

## 1b - Checkpoint Storage (Google Drive)

In [ ]:
# -- Checkpoint storage on Google Drive -------------------------------------
# Checkpoints, final models, submissions and the experiment log are written here
# so they survive a Colab disconnect and the sweep can resume across sessions.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_ROOT = '/content/drive/MyDrive/health_qa_checkpoints'
except Exception as e:
    CKPT_ROOT = './checkpoints'   # local fallback (not on Colab)
    print('(Drive not mounted, using local folder):', e)
import os
os.makedirs(CKPT_ROOT, exist_ok=True)
print('Checkpoints will be saved to:', CKPT_ROOT)

In [ ]:
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', None)

print('OK Imports complete')

## 2 - Set File Paths

In [ ]:
# Data loading is account-free and reproducible (no Google Drive needed):
#   1. Use local ./data if the files are already present (fast local runs).
#   2. Otherwise download them from the GitHub repo (works on any Colab account).
GITHUB_RAW = 'https://raw.githubusercontent.com/Nick-Lemy/multilingual-health-qa/main/data'

DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)

DATA_FILES = ['Train.csv', 'Test.csv', 'Val.csv', 'SampleSubmission.csv']
for fname in DATA_FILES:
    local = DATA_DIR / fname
    if not local.exists():
        url = f'{GITHUB_RAW}/{fname}'
        print(f'  Downloading {fname} from GitHub ...')
        pd.read_csv(url).to_csv(local, index=False)

TRAIN_PATH      = DATA_DIR / 'Train.csv'
TEST_PATH       = DATA_DIR / 'Test.csv'
VAL_PATH        = DATA_DIR / 'Val.csv'
SAMPLE_SUB_PATH = DATA_DIR / 'SampleSubmission.csv'

# Output paths for each baseline
OUTPUT_TFIDF = DATA_DIR / 'submission_tfidf_baseline.csv'
OUTPUT_LLM   = DATA_DIR / 'submission_llm_baseline.csv'

for path in [TRAIN_PATH, TEST_PATH, VAL_PATH, SAMPLE_SUB_PATH]:
    status = 'OK' if path.exists() else 'X'
    print(f'{status} {path}')

## 3 - Load and Preview the Data

In [4]:
train             = pd.read_csv(TRAIN_PATH)
test              = pd.read_csv(TEST_PATH)
val               = pd.read_csv(VAL_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

print(f'Train shape             : {train.shape}')
print(f'Test shape              : {test.shape}')
print(f'Val shape               : {val.shape}')
print(f'Sample submission shape : {sample_submission.shape}')
print()
print('Train columns:', train.columns.tolist())
print('Test columns :', test.columns.tolist())
print('Val columns  :', val.columns.tolist())

display(train.head(3))
display(test.head(3))
display(sample_submission.head(3))

Train shape             : (29815, 4)
Test shape              : (2618, 3)
Val shape               : (6686, 4)
Sample submission shape : (2618, 4)

Train columns: ['ID', 'input', 'output', 'subset']
Test columns : ['ID', 'input', 'subset']
Val columns  : ['ID', 'input', 'output', 'subset']


,ID,input,output,subset
0,ID_TR_Aka_Gha_A3B1799D,Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye w...,Mmabun betumi aboa atipɛnfo a ebia nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ so denam: Nkate fam mmoa a wɔde bɛma na w...,Aka_Gha
1,ID_TR_Aka_Gha_1C80317F,"Edinnsiananmu bɛn na nnipa a ɛsono wɔn bɔbeasu taa de di dwuma, na yɛbɛyɛ dɛn ahwɛ ahu sɛ yɛde redi dwuma yiye?","Wɔ Ghana mu no, amanmmra no gye binary gender nkutoo tom a she/he edinnsiananmu nkutoo na ɛka ho",Aka_Gha
2,ID_TR_Aka_Gha_06671AD1,Ɔkwan bɛn so na ɔbarima ne ɔbea nna a wɔtwe wɔn ho fi ho anaa nna mu adwumadi a wɔtwentwɛn so no boa ma asiane so tew?,"Sɛ wɔtwe wɔn ho fi nna mu anaasɛ wɔtwentwɛn wɔn nan ase a, ɛboa ma asiane nso tew denam asiane a ɛwɔ STI ne HIV a ɛb...",Aka_Gha


,ID,input,subset
0,ID_TS_Aka_Gha_A3B1799D,"Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwuma de asiw GBV ano ma.",Aka_Gha
1,ID_TS_Aka_Gha_1C80317F,Dɛn ne nea ebetumi afi hokwan a mmabun wɔ sɛ wonya nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu a wobu...,Aka_Gha
2,ID_TS_Aka_Gha_06671AD1,Akwan bɛn na mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' wɔ bere a as...,Aka_Gha


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
1,ID_TS_Aka_Gha_1C80317F,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
2,ID_TS_Aka_Gha_06671AD1,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"


In [ ]:
# Explore language distribution in training data
print('Language distribution in training set:')
display(train['subset'].value_counts())

Language distribution in training set:


,count
subset,
Eng_Uga,7624
Aka_Gha,4455
Eng_Gha,4443
Eng_Eth,3915
Lug_Uga,3383
Eng_Ken,2080
Swa_Ken,2070
Amh_Eth,1845


In [ ]:
ID_COL           = 'ID'
TEST_ID_COL      = 'ID'
QUESTION_COL     = 'input'
TEST_QUESTION_COL= 'input'
ANSWER_COL       = 'output'
LANG_COL         = 'subset'
TEST_LANG_COL    = 'subset'

print(f'  Train ID        : {ID_COL}')
print(f'  Test ID         : {TEST_ID_COL}')
print(f'  Train question  : {QUESTION_COL}')
print(f'  Test question   : {TEST_QUESTION_COL}')
print(f'  Train answer    : {ANSWER_COL}')
# -- Language code mapping ------------------------------------------------------
# The `subset` column encodes language and country as "<LangCode>_<CountryCode>".
# Only the first part identifies the language; the second is the country.
# This dictionary maps the language prefix to a full language name used in prompts.
SUBSET_TO_LANGUAGE = {
    'Eng': 'English',
    'Aka': 'Akan',
    'Lug': 'Luganda',
    'Swa': 'Swahili',
    'Amh': 'Amharic',
}

def subset_to_language_name(subset_code: str) -> str:
    """
    Extract the full language name from a subset code such as 'Amh_Eth' or 'Aka_Gha'.
    Falls back to the raw code if the prefix is not recognised.
    """
    if not subset_code or not isinstance(subset_code, str):
        return 'English'
    lang_prefix = subset_code.split('_')[0]
    return SUBSET_TO_LANGUAGE.get(lang_prefix, subset_code)

print('Language mapping:')
for code, name in SUBSET_TO_LANGUAGE.items():
    print(f'  {code}_* -> {name}')


## 4 - Exploratory Data Analysis (EDA)
We examine language balance, question/answer length distributions, data-quality
issues (duplicates, empties), and train/val overlap. Findings here drive concrete
modelling choices - notably the input/output length budget and per-language handling.

In [ ]:
# -- Exploratory Data Analysis ----------------------------------------------
# All EDA runs on copies so it never mutates the frames used downstream.
# We explore: (1) language balance, (2) question/answer length distributions in
# both words and characters, (3) duplicates/near-empties, (4) train/val overlap.
# The length analysis directly informs MAX_INPUT_LENGTH / MAX_OUTPUT_LENGTH and
# the "concise answers win ROUGE" strategy (ROUGE = 74% of the score).
import matplotlib.pyplot as plt

def _word_count(series):
    return series.fillna('').astype(str).str.split().map(len)

eda = train[[QUESTION_COL, ANSWER_COL, LANG_COL]].copy()
eda['language'] = eda[LANG_COL].map(subset_to_language_name)
eda['q_words']  = _word_count(eda[QUESTION_COL])
eda['a_words']  = _word_count(eda[ANSWER_COL])
eda['q_chars']  = eda[QUESTION_COL].fillna('').astype(str).str.len()
eda['a_chars']  = eda[ANSWER_COL].fillna('').astype(str).str.len()

# -- 1) Per-subset / per-language balance -----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train[LANG_COL].value_counts().plot.bar(ax=axes[0], color='#4C72B0')
axes[0].set_title('Training examples per subset (lang_country)')
axes[0].set_ylabel('count')
eda['language'].value_counts().plot.bar(ax=axes[1], color='#55A868')
axes[1].set_title('Training examples per language')
axes[1].set_ylabel('count')
plt.tight_layout(); plt.show()

# -- 2) Answer & question length distributions (words) ----------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for col, ax, title in [('q_words', axes[0], 'Question length (words)'),
                       ('a_words', axes[1], 'Answer length (words)')]:
    clip = eda[col].quantile(0.99)
    eda[col].clip(upper=clip).plot.hist(bins=50, ax=ax, color='#C44E52', alpha=0.8)
    ax.axvline(eda[col].median(), color='k', ls='--', label=f'median={eda[col].median():.0f}')
    ax.axvline(eda[col].quantile(0.95), color='b', ls=':', label=f'95th={eda[col].quantile(0.95):.0f}')
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

# -- 3) Answer length by language (boxplot) ---------------------------------
fig, ax = plt.subplots(figsize=(10, 4))
langs = sorted(eda['language'].unique())
ax.boxplot([eda.loc[eda['language'] == lg, 'a_words'].clip(upper=eda['a_words'].quantile(0.99))
            for lg in langs], showfliers=False)
ax.set_xticks(range(1, len(langs) + 1)); ax.set_xticklabels(langs, rotation=20)
ax.set_title('Answer length (words) by language'); ax.set_ylabel('words')
plt.tight_layout(); plt.show()

# -- 4) Per-language summary table ------------------------------------------
summary = eda.groupby('language').agg(
    n            = (QUESTION_COL, 'size'),
    q_words_med  = ('q_words', 'median'),
    a_words_med  = ('a_words', 'median'),
    a_words_p95  = ('a_words', lambda s: s.quantile(0.95)),
    a_chars_med  = ('a_chars', 'median'),
).round(1)
print('Per-language summary:')
display(summary)

# -- 5) Data-quality checks -------------------------------------------------
dup_q   = train[QUESTION_COL].duplicated().sum()
dup_qa  = train.duplicated(subset=[QUESTION_COL, ANSWER_COL]).sum()
empty_q = (eda['q_words'] == 0).sum()
empty_a = (eda['a_words'] == 0).sum()
val_q   = set(val[QUESTION_COL].astype(str))
overlap = len(set(train[QUESTION_COL].astype(str)) & val_q)
print(f'\nData-quality checks (train):')
print(f'  duplicate questions          : {dup_q:,}')
print(f'  duplicate (question, answer) : {dup_qa:,}')
print(f'  empty questions / answers    : {empty_q:,} / {empty_a:,}')
print(f'  train<->val identical questions : {overlap:,} of {len(val):,} val rows')

# -- 6) Length-budget recommendation (tokens ~ 1.3 x words for mT5) ----------
a95 = eda['a_words'].quantile(0.95); q95 = eda['q_words'].quantile(0.95)
print(f'\nLength-budget guidance (rough: mT5 tokens ~ 1.3 x words):')
print(f'  95th-pct question words ~ {q95:.0f}  -> MAX_INPUT_LENGTH  ~ {int(q95*1.3)+8}')
print(f'  95th-pct answer   words ~ {a95:.0f}  -> MAX_OUTPUT_LENGTH ~ {int(a95*1.3)+8}')
print('  (Capping output near the 95th pct speeds generation and discourages')
print('   verbose answers that dilute ROUGE precision.)')


## 5 - Text Cleaning

In [8]:
def clean_text(x):
    """Strip whitespace and handle null values."""
    if pd.isna(x):
        return ''
    return str(x).strip()

train[QUESTION_COL]      = train[QUESTION_COL].map(clean_text)
train[ANSWER_COL]        = train[ANSWER_COL].map(clean_text)
test[TEST_QUESTION_COL]  = test[TEST_QUESTION_COL].map(clean_text)
val[QUESTION_COL]        = val[QUESTION_COL].map(clean_text)
val[ANSWER_COL]          = val[ANSWER_COL].map(clean_text)

# Remove rows with empty questions or answers
train = train[(train[QUESTION_COL] != '') & (train[ANSWER_COL] != '')].reset_index(drop=True)
test  = test[test[TEST_QUESTION_COL]  != ''].reset_index(drop=True)
val   = val[(val[QUESTION_COL] != '') & (val[ANSWER_COL] != '')].reset_index(drop=True)

print(f'Cleaned train shape : {train.shape}')
print(f'Cleaned test shape  : {test.shape}')
print(f'Cleaned val shape   : {val.shape}')

Cleaned train shape : (29814, 4)
Cleaned test shape  : (2618, 3)
Cleaned val shape   : (6686, 4)


## 7 - Evaluation Utilities

ROUGE-1 and ROUGE-L scoring using whitespace tokenisation - safe for non-English scripts.

In [ ]:
try:
    from rouge_score import rouge_scorer

    class WhitespaceTokenizer:
        """Whitespace tokeniser - language-agnostic and safe for African scripts."""
        def tokenize(self, text):
            if text is None:
                return []
            return str(text).strip().split()

    def compute_rouge(predictions, references):
        """
        Compute mean ROUGE-1 and ROUGE-L F1 scores.

        Parameters
        ----------
        predictions : list[str]
        references  : list[str]

        Returns
        -------
        dict with rouge1_f1 and rougeL_f1
        """
        scorer = rouge_scorer.RougeScorer(
            ['rouge1', 'rougeL'],
            tokenizer    = WhitespaceTokenizer(),
            use_stemmer  = False,
        )
        r1_scores, rl_scores = [], []

        for pred, ref in zip(predictions, references):
            score = scorer.score(str(ref), str(pred))
            r1_scores.append(score['rouge1'].fmeasure)
            rl_scores.append(score['rougeL'].fmeasure)

        return {
            'rouge1_f1': float(np.mean(r1_scores)) if r1_scores else 0.0,
            'rougeL_f1': float(np.mean(rl_scores)) if rl_scores else 0.0,
        }

    def compute_rouge_by_language(predictions, references, languages):
        """Compute ROUGE scores broken down by language."""
        results = {}
        lang_arr = np.array(languages)

        for lang in np.unique(lang_arr):
            mask    = lang_arr == lang
            preds_l = [p for p, m in zip(predictions, mask) if m]
            refs_l  = [r for r, m in zip(references,  mask) if m]
            results[lang] = compute_rouge(preds_l, refs_l)

        return pd.DataFrame(results).T

    print('OK ROUGE scorer loaded')

except ImportError:
    print('WARNING  rouge-score not installed. Run: pip install rouge-score')
    compute_rouge = None

## 7b - Experiment Tracking
A lightweight logger records every meaningful run (config + validation ROUGE +
leaderboard score) to `data/experiments_log.csv`. Call `log_experiment(...)` after
each experiment, then `show_experiments()` / `plot_experiment_progression()` to
review progress. `val_proxy = 0.37-ROUGE-1 + 0.37-ROUGE-L` approximates the official
score offline (the 0.26 LLM-judge term is only available on the leaderboard).

In [ ]:
# -- Experiment tracking ----------------------------------------------------
# A tiny, dependency-free experiment logger so every meaningful run is recorded
# with its config, validation ROUGE, and (once submitted) the Zindi leaderboard
# score. This makes the >=10-experiment requirement systematic and defensible:
# each row answers "what changed / why / outcome".
#
# Columns: timestamp, name, val_rouge1, val_rougeL, val_proxy, leaderboard, notes, config
#   - val_proxy = 0.37*ROUGE-1 + 0.37*ROUGE-L - the ROUGE portion of the official
#     metric (the 0.26 LLM-judge term can't be computed locally), so it is a
#     useful offline proxy for ranking experiments before spending a submission.
import json as _json
from datetime import datetime

EXPERIMENTS_LOG = Path(CKPT_ROOT) / 'experiments_log.csv'

_LOG_COLUMNS = ['timestamp', 'name', 'val_rouge1', 'val_rougeL',
                'val_proxy', 'leaderboard', 'notes', 'config']


def _load_log():
    if Path(EXPERIMENTS_LOG).exists():
        return pd.read_csv(EXPERIMENTS_LOG)
    return pd.DataFrame(columns=_LOG_COLUMNS)


def log_experiment(name, config=None, val_rouge1=None, val_rougeL=None,
                   leaderboard=None, notes=''):
    """
    Append one experiment to data/experiments_log.csv and return the full log.

    Parameters
    ----------
    name        : short unique label, e.g. 'mt5-small-ft-ep3'
    config      : dict of the settings that define this run (model, gen params...)
    val_rouge1  : mean validation ROUGE-1 F1 (optional)
    val_rougeL  : mean validation ROUGE-L F1 (optional)
    leaderboard : Zindi public leaderboard score once submitted (optional)
    notes       : free-text 'why / insight'
    """
    log = _load_log()
    proxy = None
    if val_rouge1 is not None and val_rougeL is not None:
        proxy = round(0.37 * val_rouge1 + 0.37 * val_rougeL, 4)
    row = {
        'timestamp'  : datetime.now().strftime('%Y-%m-%d %H:%M'),
        'name'       : name,
        'val_rouge1' : round(val_rouge1, 4) if val_rouge1 is not None else None,
        'val_rougeL' : round(val_rougeL, 4) if val_rougeL is not None else None,
        'val_proxy'  : proxy,
        'leaderboard': leaderboard,
        'notes'      : notes,
        'config'     : _json.dumps(config, ensure_ascii=False) if config else '',
    }
    # Replace any prior row with the same name (re-runs overwrite cleanly).
    log = log[log['name'] != name]
    log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    log.to_csv(EXPERIMENTS_LOG, index=False, encoding='utf-8')
    print(f'OK Logged experiment "{name}"  (val_proxy={proxy})')
    return log


def show_experiments(sort_by='val_proxy'):
    """Display the experiment log, best first."""
    log = _load_log()
    if log.empty:
        print('No experiments logged yet.')
        return log
    view = log.sort_values(sort_by, ascending=False, na_position='last')
    display(view[['name', 'val_rouge1', 'val_rougeL', 'val_proxy', 'leaderboard', 'notes']])
    return view


def plot_experiment_progression():
    """Plot validation proxy and leaderboard score across experiments (in log order)."""
    import matplotlib.pyplot as plt
    log = _load_log()
    if log.empty:
        print('No experiments to plot yet.')
        return
    x = range(len(log))
    fig, ax = plt.subplots(figsize=(11, 4))
    if log['val_proxy'].notna().any():
        ax.plot(x, log['val_proxy'], 'o-', label='val proxy (0.37-R1+0.37-RL)')
    if log['leaderboard'].notna().any():
        ax.plot(x, pd.to_numeric(log['leaderboard'], errors='coerce'),
                's--', label='Zindi leaderboard')
    ax.set_xticks(list(x)); ax.set_xticklabels(log['name'], rotation=30, ha='right')
    ax.set_ylabel('score'); ax.set_title('Experiment progression')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()


print('OK Experiment tracker ready - use log_experiment(), show_experiments(), plot_experiment_progression()')
print(f'   Log file: {EXPERIMENTS_LOG}')


## 8 - Baseline 1: TF-IDF Retrieval

For each test question, find the most similar training question using TF-IDF character n-grams and return its answer.

**Why character n-grams?** Character-level features work across scripts (Latin, Amharic Ge'ez, etc.) without requiring language-specific tokenisation.

In [ ]:
class TfidfRetrievalAnswerer:
    """
    TF-IDF nearest-neighbour retrieval baseline.

    Builds a per-language model if a group column is available,
    falling back to a global model for unseen groups.
    """

    def __init__(self, question_col, answer_col, group_col=None,
                 ngram_range=(3, 5), max_features=200_000):
        self.question_col = question_col
        self.answer_col   = answer_col
        self.group_col    = group_col
        self.ngram_range  = ngram_range
        self.max_features = max_features
        self.models       = {}
        self.global_model = None

    def _fit_single(self, df):
        """Fit a vectoriser and nearest-neighbour index on a subset."""
        vectorizer = TfidfVectorizer(
            analyzer     = 'char_wb',
            ngram_range  = self.ngram_range,
            min_df       = 1,
            max_features = self.max_features,
            lowercase    = False,   # preserve case for non-Latin scripts
        )
        questions = df[self.question_col].fillna('').astype(str).tolist()
        answers   = df[self.answer_col].fillna('').astype(str).tolist()

        X  = vectorizer.fit_transform(questions)
        nn = NearestNeighbors(n_neighbors=1, metric='cosine')
        nn.fit(X)

        return {
            'vectorizer': vectorizer,
            'nn'        : nn,
            'answers'   : np.array(answers,   dtype=object),
            'questions' : np.array(questions, dtype=object),
        }

    def fit(self, df):
        """Fit the global model and per-group models."""
        self.global_model = self._fit_single(df)
        if self.group_col and self.group_col in df.columns:
            for group, sub in df.groupby(self.group_col):
                if len(sub) >= 2:
                    self.models[group] = self._fit_single(sub)
        print(f'  Fitted global model + {len(self.models)} group model(s)')
        return self

    def _predict_one_from_model(self, question, model):
        Xq        = model['vectorizer'].transform([question])
        dist, idx = model['nn'].kneighbors(Xq, n_neighbors=1)
        i         = idx[0][0]
        sim       = 1 - float(dist[0][0])
        return model['answers'][i], sim, model['questions'][i]

    def predict_one(self, question, group=None):
        model = self.models.get(group, self.global_model) if group is not None else self.global_model
        return self._predict_one_from_model(question, model)

    def predict(self, df, question_col, group_col=None):
        outputs, similarities, matched = [], [], []
        for _, row in df.iterrows():
            question = clean_text(row[question_col])
            group    = row[group_col] if group_col and group_col in df.columns else None
            answer, sim, matched_q = self.predict_one(question, group)
            outputs.append(answer)
            similarities.append(sim)
            matched.append(matched_q)
        return outputs, similarities, matched

print('OK TfidfRetrievalAnswerer defined')

In [ ]:
# Choose grouping strategy - prefer config (language+country), else language
GROUP_COL      =  LANG_COL
TEST_GROUP_COL =  TEST_LANG_COL

print(f'Group column      : {GROUP_COL}')
print(f'Test group column : {TEST_GROUP_COL}')

In [ ]:
# -- Validate TF-IDF baseline on the local validation set ----------------------
print('Training TF-IDF retrieval baseline on training partition...')

answerer_valid = TfidfRetrievalAnswerer(
    question_col = QUESTION_COL,
    answer_col   = ANSWER_COL,
    group_col    = GROUP_COL,
).fit(train)

valid_pred, valid_sim, valid_match = answerer_valid.predict(
    val,
    question_col = QUESTION_COL,
    group_col    = GROUP_COL,
)

if compute_rouge:
    metrics = compute_rouge(valid_pred, val[ANSWER_COL].tolist())
    print(f'\n TF-IDF Baseline - Validation ROUGE Scores')
    print(f'   ROUGE-1 F1 : {metrics["rouge1_f1"]:.4f}')
    print(f'   ROUGE-L F1 : {metrics["rougeL_f1"]:.4f}')

    # Break down by language
    if LANG_COL and LANG_COL in val.columns:
        print('\n ROUGE scores by language:')
        lang_metrics = compute_rouge_by_language(
            valid_pred,
            val[ANSWER_COL].tolist(),
            val[LANG_COL].tolist()
        )
        display(lang_metrics.round(4))

# Preview
preview = val[[ID_COL, QUESTION_COL, ANSWER_COL]].copy()
preview['baseline_answer']        = valid_pred
preview['retrieval_similarity']   = [f'{s:.3f}' for s in valid_sim]
preview['matched_train_question'] = valid_match
display(preview.head(5))

# -- Log this experiment ----------------------------------------------------
if compute_rouge:
    log_experiment(
        name        = 'tfidf-retrieval',
        config      = {'method': 'tfidf-char-ngram', 'ngram_range': '(3,5)',
                       'group_col': GROUP_COL},
        val_rouge1  = metrics['rouge1_f1'],
        val_rougeL  = metrics['rougeL_f1'],
        notes       = 'Baseline 1: per-language TF-IDF nearest-neighbour retrieval',
    )

In [ ]:
# -- Train on all data and predict test answers ---------------------------------
print('Training TF-IDF retrieval on full training set...')

answerer = TfidfRetrievalAnswerer(
    question_col = QUESTION_COL,
    answer_col   = ANSWER_COL,
    group_col    = GROUP_COL,
).fit(train)

test_pred_tfidf, test_sim, test_match = answerer.predict(
    test,
    question_col = TEST_QUESTION_COL,
    group_col    = TEST_GROUP_COL,
)

print(f'Generated {len(test_pred_tfidf)} predictions')

preview = test[[TEST_ID_COL, TEST_QUESTION_COL]].copy()
preview['baseline_answer']      = test_pred_tfidf
preview['retrieval_similarity'] = [f'{s:.3f}' for s in test_sim]
display(preview.head(5))

## 9 - Baseline 2: Multilingual LLM (mT5 / AfroLM)

This baseline uses a pre-trained multilingual sequence-to-sequence model to generate answers directly, without retrieval.

**Model options (choose one):**

| Model | Languages | Size | Notes |
|---|---|---|---|
| `google/mt5-small` | 101 languages incl. Swahili | 300M | Fast, good multilingual coverage |
| `google/mt5-base` | 101 languages | 580M | Better quality, slower |
| `facebook/nllb-200-distilled-600M` | 200 languages incl. Luganda, Akan | 600M | Best African language coverage |
| `Helsinki-NLP/opus-mt-mul-en` | Many -> English | 300M | English output only |

> **Recommendation:** Start with `google/mt5-small` for speed, then try `facebook/nllb-200-distilled-600M` for better coverage of low-resource languages.

**For a strong fine-tuned solution:**
Fine-tune `google/mt5-base` or `facebook/nllb-200-distilled-600M` on the provided training data using the Hugging Face `Seq2SeqTrainer`. See the fine-tuning section at the end of this notebook.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -- Model selection ------------------------------------------------------------
# mT5 is a text-to-text model well suited to *same-language* answer generation,
# which is exactly this task. Default to mt5-small for fast iteration, then switch
# to mt5-base for higher quality once the pipeline is validated end-to-end.
#
# NOTE: facebook/nllb-200-distilled-600M is a *translation* model - it maps text
# from a source language into a *different* target language, so it is the wrong
# base for generating an answer in the SAME language as the question. It is kept
# here only as a candidate experiment, never as the default.
MODEL_NAME = 'google/mt5-small'       # fast first run
# MODEL_NAME = 'google/mt5-base'      # better quality (slower, more VRAM)
# MODEL_NAME = 'facebook/nllb-200-distilled-600M'  # experiment only (translation model)

MAX_INPUT_LENGTH  = 128    # tokens for input question  (EDA: 95th-pct ~ 52 tok)
MAX_OUTPUT_LENGTH = 256    # tokens for generated answer (EDA: 95th-pct ~ 247 tok)
BATCH_SIZE_LLM    = 8      # Reduce to 4 if you get out-of-memory errors
NUM_BEAMS         = 4      # Beam search width - higher = better quality, slower
LENGTH_PENALTY       = 1.0   # <1 favours shorter outputs, >1 longer (decoding knob)
NO_REPEAT_NGRAM_SIZE = 3     # block repeated n-grams during generation
USE_LANG_PROMPT      = True  # prepend 'Answer ... in <language>:'; experiments toggle this

# Use GPU if available
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
print(f'Model   : {MODEL_NAME}')

if DEVICE == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# -- Load the model and tokeniser -----------------------------------------------
print(f'Loading {MODEL_NAME}...')
print('This may take a few minutes on first run (downloading model weights).')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_llm = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    # Always load in float32 so gradient computation stays in float32.
    # fp16/bf16 mixed precision is handled by the Trainer via grad scaler,
    # not by storing the model weights in float16 directly.
    torch_dtype = torch.float32,
)
model_llm = model_llm.to(DEVICE)
model_llm.eval()

print(f'OK {MODEL_NAME} loaded on {DEVICE}')
print(f'   Parameters : {sum(p.numel() for p in model_llm.parameters()) / 1e6:.0f}M')

In [ ]:
import re

def build_prompt(question: str, language: str = None) -> str:
    """
    Build an input prompt for the model.

    For mT5: prefix the question with a task description.
    The model learns to associate the prefix with the generation task.

    `language` may be a raw subset code (e.g. 'Amh_Eth') or a full language
    name. It is resolved through `subset_to_language_name` so the model always
    receives a human-readable language name in the prompt rather than an opaque
    code.

    Parameters
    ----------
    question : str
        The health question to answer.
    language : str, optional
        Subset code (e.g. 'Amh_Eth') or full language name. Resolved to a
        human-readable name before being inserted into the prompt.

    Returns
    -------
    str
    """
    q = str(question).strip()
    if not USE_LANG_PROMPT:
        return f'Answer this health question: {q}'
    lang_name = subset_to_language_name(language) if language else None
    if lang_name:
        return f'Answer this health question in {lang_name}: {q}'
    return f'Answer this health question: {q}'


def generate_answers_batch(questions: list, languages: list = None,
                           batch_size: int = BATCH_SIZE_LLM) -> list:
    """
    Generate answers for a list of questions using the loaded LLM.

    Processes questions in batches to avoid out-of-memory errors.

    Parameters
    ----------
    questions : list[str]
    languages : list[str], optional
    batch_size : int

    Returns
    -------
    list[str]
    """
    if languages is None:
        languages = [None] * len(questions)

    all_answers = []
    n_batches   = (len(questions) + batch_size - 1) // batch_size

    for batch_idx in range(n_batches):
        start = batch_idx * batch_size
        end   = min(start + batch_size, len(questions))

        batch_questions = questions[start:end]
        batch_languages = languages[start:end]

        # Build prompts
        prompts = [
            build_prompt(q, l)
            for q, l in zip(batch_questions, batch_languages)
        ]

        # Tokenise
        inputs = tokenizer(
            prompts,
            return_tensors = 'pt',
            padding        = True,
            truncation     = True,
            max_length     = MAX_INPUT_LENGTH,
        ).to(DEVICE)

        # Generate
        with torch.no_grad():
            outputs = model_llm.generate(
                **inputs,
                max_new_tokens  = MAX_OUTPUT_LENGTH,
                num_beams       = NUM_BEAMS,
                early_stopping  = True,
                no_repeat_ngram_size = NO_REPEAT_NGRAM_SIZE,
                length_penalty  = LENGTH_PENALTY,
            )

        # Decode
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # Post-process: strip mT5 sentinel tokens (<extra_id_N>) that the
        # model may emit when it has not been fine-tuned on a seq2seq task.
        # mT5 is pre-trained with a span-corruption objective that uses these
        # tokens as placeholders; a zero-shot prompt may trigger them because
        # the model has never been trained to suppress them in open generation.
        cleaned = [re.sub(r'<extra_id_\d+>', '', ans).strip() for ans in decoded]
        all_answers.extend(cleaned)

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == n_batches:
            print(f'  Batch {batch_idx + 1}/{n_batches} - {end}/{len(questions)} questions processed')

    return all_answers

print('OK LLM generation functions defined')

In [ ]:
# -- Quick sanity check on a few examples --------------------------------------
print('Running sanity check on 3 validation examples...')

sample     = val.head(3)
q_sample   = sample[QUESTION_COL].tolist()
l_sample   = sample[LANG_COL].tolist() if LANG_COL else None
ref_sample = sample[ANSWER_COL].tolist()

gen_sample = generate_answers_batch(q_sample, l_sample, batch_size=3)

for i, (q, ref, gen) in enumerate(zip(q_sample, ref_sample, gen_sample)):
    lang = l_sample[i] if l_sample else 'unknown'
    print(f'\n[{i+1}] Language : {lang}')
    print(f'    Question  : {q[:120]}')
    print(f'    Reference : {ref[:120]}')
    print(f'    Generated : {gen[:120]}')

In [ ]:
# -- Validate LLM baseline on the local validation set -------------------------
# Note: this cell can take several minutes depending on your hardware.
# Reduce VALIDATION_SAMPLE_SIZE to speed it up during experimentation.

VALIDATION_SAMPLE_SIZE = 200   # Set to None to evaluate on the full validation set

if VALIDATION_SAMPLE_SIZE:
    val_sample = val.sample(
        n            = min(VALIDATION_SAMPLE_SIZE, len(val)),
        random_state = SEED
    )
else:
    val_sample = val

print(f'Evaluating LLM baseline on {len(val_sample)} validation examples...')

val_questions = val_sample[QUESTION_COL].tolist()
val_languages = val_sample[LANG_COL].tolist() if LANG_COL else None
val_references= val_sample[ANSWER_COL].tolist()

val_predictions_llm = generate_answers_batch(val_questions, val_languages)

if compute_rouge:
    metrics_llm = compute_rouge(val_predictions_llm, val_references)
    print(f'\n LLM Baseline - Validation ROUGE Scores ({MODEL_NAME})')
    print(f'   ROUGE-1 F1 : {metrics_llm["rouge1_f1"]:.4f}')
    print(f'   ROUGE-L F1 : {metrics_llm["rougeL_f1"]:.4f}')

    if LANG_COL and LANG_COL in val_sample.columns:
        print('\n ROUGE scores by language (LLM baseline):')
        lang_metrics_llm = compute_rouge_by_language(
            val_predictions_llm,
            val_references,
            val_sample[LANG_COL].tolist()
        )
        display(lang_metrics_llm.round(4))

# -- Log this experiment ----------------------------------------------------
if compute_rouge:
    log_experiment(
        name        = f'zeroshot-{MODEL_NAME.split("/")[-1]}',
        config      = {'model': MODEL_NAME, 'mode': 'zero-shot',
                       'num_beams': NUM_BEAMS, 'max_output': MAX_OUTPUT_LENGTH},
        val_rouge1  = metrics_llm['rouge1_f1'],
        val_rougeL  = metrics_llm['rougeL_f1'],
        notes       = 'Baseline 2: pre-trained LLM, no fine-tuning (expected to be weak)',
    )

## 10 - Create Submission Files

Each submission must contain exactly four columns: `ID`, `TargetRLF1`, `TargetR1F1`, `TargetLLM`.
All three target columns should contain the same generated answer.

In [ ]:
SUBMISSION_FALLBACK = 'Please consult a healthcare professional.'

def make_submission(ids, predictions, output_path, fallback=SUBMISSION_FALLBACK):
    """
    Build and save a valid Zindi submission.

    Guarantees EVERY required test ID appears exactly once with a NON-EMPTY
    answer. Zindi reports blank or absent answers as
    "Rougelf1 error: Missing entries for IDs ...", so empty model outputs and
    any IDs with no prediction are filled with `fallback`.
    """
    # Strip mT5 sentinel tokens, then replace empty outputs with the fallback.
    clean = [re.sub(r'<extra_id_\d+>', '', str(p)).strip() for p in predictions]
    n_empty = sum(1 for p in clean if not p)
    clean = [p if p else fallback for p in clean]

    pred_df = pd.DataFrame({'ID': list(ids), 'answer': clean}).drop_duplicates('ID', keep='first')

    # Align to the OFFICIAL id list so nothing is missing and order matches.
    try:
        required = sample_submission[['ID']].copy()
    except NameError:
        required = pd.read_csv(SAMPLE_SUB_PATH)[['ID']].copy()
    merged = required.merge(pred_df, on='ID', how='left')
    n_missing = int(merged['answer'].isna().sum())
    merged['answer'] = merged['answer'].fillna(fallback)
    merged.loc[merged['answer'].str.strip() == '', 'answer'] = fallback

    sub = pd.DataFrame({
        'ID'        : merged['ID'],
        'TargetRLF1': merged['answer'],
        'TargetR1F1': merged['answer'],
        'TargetLLM' : merged['answer'],
    })

    # Checks: complete, unique, non-empty
    assert list(sub.columns) == ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']
    assert len(sub) == len(required), f'{len(sub)} rows vs {len(required)} required IDs'
    assert sub['ID'].nunique() == len(sub), 'duplicate IDs in submission'
    assert (sub['TargetRLF1'].str.strip() != '').all(), 'blank answers remain'
    assert sub[['TargetRLF1', 'TargetR1F1', 'TargetLLM']].notna().all().all()

    sub.to_csv(output_path, index=False, encoding='utf-8')
    if n_empty:
        print(f'  WARNING: {n_empty} empty prediction(s) replaced with fallback')
    if n_missing:
        print(f'  WARNING: {n_missing} required ID(s) had no prediction -> fallback')
    print(f'Submission saved: {output_path}  shape={sub.shape}')
    display(sub.head(3))
    return sub


# Save the TF-IDF baseline submission (complete + non-empty -> safe to submit).
print('Saving TF-IDF baseline submission...')
sub_tfidf = make_submission(test[TEST_ID_COL].values, test_pred_tfidf, OUTPUT_TFIDF)


## 11 - Compare Baselines

In [ ]:
if compute_rouge:
    # Recompute TF-IDF validation scores for comparison
    tfidf_preds_val, _, _ = answerer_valid.predict(
        val_sample, question_col=QUESTION_COL, group_col=GROUP_COL
    )
    metrics_tfidf = compute_rouge(tfidf_preds_val, val_references)

    comparison = pd.DataFrame({
        'Baseline'  : ['TF-IDF Retrieval', f'LLM ({MODEL_NAME})'],
        'ROUGE-1 F1': [metrics_tfidf['rouge1_f1'], metrics_llm['rouge1_f1']],
        'ROUGE-L F1': [metrics_tfidf['rougeL_f1'], metrics_llm['rougeL_f1']],
    })

    print(' Baseline Comparison (validation set):')
    display(comparison.round(4))
    print()
    print('Note: The LLM score shown here is for a zero-shot (untrained) model.')
    print('Fine-tuning on the training data will significantly improve this.')

## 12 - Fine-tuning & Experiments

Manual single-model fine-tuning has been consolidated into the **automated runner in
section 14**, which trains, checkpoints to Google Drive, evaluates, writes a submission, and
logs every experiment. Continue to section 14 - there is nothing to run in this section.

## 13 - Experiment Roadmap (>=10 meaningful experiments)

The rubric requires **at least 10 meaningful experiments**, each documented with *what
changed / why / outcome / insight*. Use `log_experiment(...)` for every run so the table
and progression plot build automatically. Candidate experiments (mix of CPU-cheap and
GPU runs):

| # | Experiment | What changes | Why |
|---|------------|--------------|-----|
| 1 | TF-IDF retrieval baseline | char n-gram retrieval | establish a non-neural floor |
| 2 | Zero-shot mT5 | no fine-tuning | show why fine-tuning is needed |
| 3 | Fine-tune mT5-small | full FT, 3 epochs | first real model |
| 4 | Length budget | MAX_OUTPUT 512 -> ~256 (EDA-driven) | speed + ROUGE precision |
| 5 | Prompt variant | with/without language prefix in `build_prompt` | test language conditioning |
| 6 | Generation params | beams, `length_penalty`, `no_repeat_ngram_size` | tune decoding for ROUGE |
| 7 | mT5-small -> mT5-base | larger backbone | quality vs cost trade-off |
| 8 | Epochs / LR sweep | e.g. 3->5 epochs, 5e-5->3e-4 | find best training schedule |
| 9 | Dedup / cleaning | drop 276 duplicate Q&A, 1,469 dup questions | cleaner training signal |
| 10 | Per-language vs global | separate handling for weak langs (Amharic) | address per-language gaps |
| 11 | LoRA / PEFT | parameter-efficient fine-tune | cheaper, comparable quality |
| 12 | Retrieval + LLM hybrid | back off to TF-IDF when generation is low-confidence | combine strengths |

**For each run, after computing validation ROUGE, call `log_experiment(...)`, submit to
Zindi, then set the `leaderboard=` value on the next call (or edit the CSV).**

## 14 - Automated Experiments (run the whole sweep here)

Everything below is driven by `run_experiment(cfg)`. Each experiment trains
(**checkpointing every epoch to Google Drive**), **resumes** if interrupted, **skips**
if already finished, evaluates on validation, writes a submission to Drive, and logs it.

**Just run top to bottom:**
1. Run the runner cell.
2. Run the **"Run all experiments"** cell - it loops through all 10. If Colab
   disconnects, run that cell again: finished experiments are skipped and an
   interrupted one resumes from its last epoch.
3. `FAST_MODE=True` screens each run on an 8k subset. After picking the winner from
   `show_experiments()`, set `FAST_MODE=False` and run the final cell to full-train it.
4. After each Zindi upload, record the score: `log_experiment(name='<tag>', leaderboard=<score>)`.

In [ ]:
# -- Automated experiment runner (Drive checkpoints - resume - skip-if-done) --
# One config dict = one experiment. run_experiment() trains (checkpointing EVERY
# epoch to Google Drive), resumes from the last checkpoint if interrupted, skips
# entirely if a finished model already exists on Drive, evaluates on validation,
# writes a submission, and logs the result.
import os, torch
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq)
from transformers.trainer_utils import get_last_checkpoint
from datasets import Dataset
from sklearn.model_selection import train_test_split

# FAST_MODE screens each experiment on a training subset so the whole sweep fits
# in a session or two. Set FAST_MODE=False to full-train your chosen winner.
FAST_MODE       = True
FAST_TRAIN_ROWS = 8000
FAST_VAL_ROWS   = 400
SKIP_IF_DONE    = True   # skip retraining if a finished model already exists on Drive


def _build_ds(df, tok, max_in, max_out):
    recs = [{'prompt': build_prompt(str(r[QUESTION_COL]), str(r[LANG_COL])),
             'answer': str(r[ANSWER_COL])} for _, r in df.iterrows()]
    ds = Dataset.from_list(recs)
    def prep(ex):
        mi  = tok(ex['prompt'], max_length=max_in, truncation=True, padding=False)
        lab = tok(text_target=ex['answer'], max_length=max_out, truncation=True, padding=False)
        mi['labels'] = [[(t if t != tok.pad_token_id else -100) for t in s]
                        for s in lab['input_ids']]
        return mi
    return ds.map(prep, batched=True, remove_columns=['prompt', 'answer'])


def run_experiment(cfg):
    """Train + evaluate + submit + log one experiment defined by a config dict."""
    global MODEL_NAME, tokenizer, model_llm
    global MAX_INPUT_LENGTH, MAX_OUTPUT_LENGTH, NUM_BEAMS
    global LENGTH_PENALTY, NO_REPEAT_NGRAM_SIZE, USE_LANG_PROMPT

    name = cfg['name']
    tag  = name + ('' if FAST_MODE else '-full')
    print(f'\n{"="*70}\n> EXPERIMENT: {tag}\n{"="*70}')

    # apply config to the shared globals the helper functions read
    MODEL_NAME           = cfg.get('model_name', 'google/mt5-small')
    MAX_INPUT_LENGTH     = cfg.get('max_input', 128)
    MAX_OUTPUT_LENGTH    = cfg.get('max_target', 256)
    NUM_BEAMS            = cfg.get('num_beams', 4)
    LENGTH_PENALTY       = cfg.get('length_penalty', 1.0)
    NO_REPEAT_NGRAM_SIZE = cfg.get('no_repeat_ngram_size', 3)
    USE_LANG_PROMPT      = cfg.get('use_lang_prompt', True)

    final_dir = os.path.join(CKPT_ROOT, f'model-{tag}')
    ckpt_dir  = os.path.join(CKPT_ROOT, f'ckpt-{tag}')

    if SKIP_IF_DONE and os.path.exists(os.path.join(final_dir, 'config.json')):
        print(f'  OK already trained - loading {final_dir}')
        tokenizer = AutoTokenizer.from_pretrained(final_dir)
        model_llm = AutoModelForSeq2SeqLM.from_pretrained(final_dir, torch_dtype=torch.float32).to(DEVICE)
    else:
        # training data (optional dedup / subsample)
        tr = train.copy()
        if cfg.get('dedup'):
            n0 = len(tr)
            tr = tr.drop_duplicates(subset=[QUESTION_COL, ANSWER_COL]).reset_index(drop=True)
            print(f'  dedup: {n0:,} -> {len(tr):,} rows')
        n_sub = cfg.get('subsample', FAST_TRAIN_ROWS if FAST_MODE else None)
        if n_sub:
            tr = tr.sample(n=min(n_sub, len(tr)), random_state=SEED).reset_index(drop=True)
            print(f'  subsample: {len(tr):,} train rows (FAST_MODE={FAST_MODE})')

        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model_llm = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(DEVICE)
        if cfg.get('lora'):
            from peft import LoraConfig, get_peft_model, TaskType
            model_llm = get_peft_model(model_llm, LoraConfig(
                task_type=TaskType.SEQ_2_SEQ_LM, r=cfg.get('lora_r', 16),
                lora_alpha=cfg.get('lora_alpha', 32), lora_dropout=0.05,
                target_modules=['q', 'v']))
            model_llm.print_trainable_parameters()

        tr_df, va_df = train_test_split(
            tr, test_size=0.05, random_state=SEED,
            stratify=tr[LANG_COL] if LANG_COL in tr.columns else None)
        ds_tr = _build_ds(tr_df, tokenizer, MAX_INPUT_LENGTH, MAX_OUTPUT_LENGTH)
        ds_va = _build_ds(va_df, tokenizer, MAX_INPUT_LENGTH, MAX_OUTPUT_LENGTH)

        args = Seq2SeqTrainingArguments(
            output_dir                  = ckpt_dir,
            num_train_epochs            = cfg.get('epochs', 2),
            per_device_train_batch_size = cfg.get('batch_size', 8),
            per_device_eval_batch_size  = cfg.get('batch_size', 8),
            learning_rate               = cfg.get('lr', 5e-5),
            predict_with_generate       = True,
            bf16                        = (DEVICE == 'cuda' and torch.cuda.is_bf16_supported()),
            fp16                        = False,
            eval_strategy               = 'epoch',
            save_strategy               = 'epoch',     # checkpoint to Drive EVERY epoch
            save_total_limit            = cfg.get('save_total_limit', 2),
            load_best_model_at_end      = True,
            metric_for_best_model       = 'eval_loss',
            generation_max_length       = MAX_OUTPUT_LENGTH,
            generation_num_beams        = 1,           # greedy eval for speed
            logging_steps               = 100,
            report_to                   = 'none',
        )
        trainer = Seq2SeqTrainer(
            model=model_llm, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
            processing_class=tokenizer,
            data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_llm,
                                                 label_pad_token_id=-100, pad_to_multiple_of=8))
        last = get_last_checkpoint(ckpt_dir) if os.path.isdir(ckpt_dir) else None
        if last:
            print(f'  (resume) resuming from {last}')
        trainer.train(resume_from_checkpoint=last)
        trainer.save_model(final_dir); tokenizer.save_pretrained(final_dir)
        print(f'  OK saved model to Drive: {final_dir}')

    # evaluate on validation
    model_llm.eval()
    vs = val.sample(
        n=min(cfg.get('val_sample', FAST_VAL_ROWS if FAST_MODE else len(val)), len(val)),
        random_state=SEED)
    vp = generate_answers_batch(vs[QUESTION_COL].tolist(), vs[LANG_COL].tolist())
    m  = compute_rouge(vp, vs[ANSWER_COL].tolist())
    print(f'  VAL  ROUGE-1={m["rouge1_f1"]:.4f}  ROUGE-L={m["rougeL_f1"]:.4f}')

    # test predictions + submission (saved to Drive so it persists)
    tp = generate_answers_batch(test[TEST_QUESTION_COL].tolist(), test[TEST_LANG_COL].tolist())
    make_submission(test[TEST_ID_COL].values, tp, os.path.join(CKPT_ROOT, f'submission_{tag}.csv'))

    log_experiment(name=tag, config=cfg,
                   val_rouge1=m['rouge1_f1'], val_rougeL=m['rougeL_f1'],
                   notes=cfg.get('notes', ''))
    return m


print('OK run_experiment ready | FAST_MODE =', FAST_MODE, '| checkpoints ->', CKPT_ROOT)


### Experiment 1: Baseline fine-tune (mT5-small)

- WHAT: Full fine-tune of mT5-small with the language-prefixed prompt.
- WHY: Reference point that every other experiment is compared against.

In [ ]:
run_experiment({
    'name': 'ft-baseline',
    'notes': 'Baseline fine-tune mt5-small, language-prefixed prompt.',
})

### Experiment 2: Prompt variant - no language prefix

- WHAT: Same as the baseline but build_prompt drops the 'in <language>' phrase.
- WHY: Tests whether telling the model the target language actually helps.

In [ ]:
run_experiment({
    'name': 'no-lang-prompt',
    'use_lang_prompt': False,
    'notes': 'Plain prompt, no language name.',
})

### Experiment 3: Preprocessing - de-duplicate Q&A

- WHAT: Remove duplicate (question, answer) pairs before training.
- WHY: Cleaner training signal and less memorisation of repeated rows.

In [ ]:
run_experiment({
    'name': 'dedup',
    'dedup': True,
    'notes': 'Drop duplicate question/answer pairs.',
})

### Experiment 4: Length budget - shorter answers

- WHAT: Cap generated answers at 128 tokens instead of 256.
- WHY: Concise answers tend to raise ROUGE precision (74 percent of the score).

In [ ]:
run_experiment({
    'name': 'shortout-128',
    'max_target': 128,
    'notes': 'Cap answer length at 128 tokens.',
})

### Experiment 5: Decoding - greedy vs beam search

- WHAT: Generate with greedy decoding (num_beams = 1).
- WHY: Faster, and checks whether beam search actually improves ROUGE here.

In [ ]:
run_experiment({
    'name': 'greedy',
    'num_beams': 1,
    'notes': 'Greedy decoding instead of 4 beams.',
})

### Experiment 6: Decoding - length penalty and repeat block

- WHAT: Set length_penalty = 0.8 and no_repeat_ngram_size = 2.
- WHY: Trims padding and repetition that dilute ROUGE.

In [ ]:
run_experiment({
    'name': 'lenpen08',
    'length_penalty': 0.8,
    'no_repeat_ngram_size': 2,
    'notes': 'Length penalty 0.8 plus no-repeat 2.',
})

### Experiment 7: Schedule - more epochs

- WHAT: Train for 4 epochs instead of 2.
- WHY: Checks for under-fitting versus over-fitting.

In [ ]:
run_experiment({
    'name': 'epochs4',
    'epochs': 4,
    'notes': 'Train 4 epochs.',
})

### Experiment 8: Schedule - higher learning rate

- WHAT: Raise the learning rate from 5e-5 to 3e-4.
- WHY: mT5 often needs a higher learning rate to move at all.

In [ ]:
run_experiment({
    'name': 'lr3e-4',
    'lr': 3e-4,
    'notes': 'Learning rate 3e-4.',
})

### Experiment 9: Bigger backbone (mT5-base)

- WHAT: Switch the model to google/mt5-base (batch size reduced to 4).
- WHY: Tests the capacity versus cost trade-off of a larger model.

In [ ]:
run_experiment({
    'name': 'mt5-base',
    'model_name': 'google/mt5-base',
    'batch_size': 4,
    'notes': 'mT5-base backbone.',
})

### Experiment 10: Parameter-efficient fine-tuning (LoRA)

- WHAT: Train LoRA adapters instead of the full model.
- WHY: Much cheaper to train and often matches full fine-tuning.

In [ ]:
run_experiment({
    'name': 'lora',
    'lora': True,
    'lr': 3e-4,
    'epochs': 3,
    'notes': 'LoRA adapters, lr 3e-4, 3 epochs.',
})

### Review all experiments
Ranked table and progression plot of everything logged so far.

In [ ]:
show_experiments()
plot_experiment_progression()

### Final: full-train the winning config
Pick the best 'name' from the table above, paste its config below, then run this cell. FAST_MODE is turned off so it trains on the full dataset for your best leaderboard submission.

In [ ]:
FAST_MODE = False            # use the full training set + full validation
run_experiment({
    'name': 'final-best',
    'model_name': 'google/mt5-small', 'epochs': 3, 'lr': 5e-5,
    'max_target': 256, 'num_beams': 4, 'use_lang_prompt': True, 'dedup': True,
    'notes': 'Full-data train of the best screened config.',
})
print('Upload this file to Zindi:', os.path.join(CKPT_ROOT, 'submission_final-best-full.csv'))